## Load data from database

In [2]:
import mysql.connector
import numpy as np
import tensorflow as tf

DATABASE_NAME = "quy_dinh_tdtu"

def connect():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="root",
        database=DATABASE_NAME
    )

def load_data_from_database(queries: list[str]) -> dict[str, list[dict]]:
    connection = connect()
    cursor = connection.cursor(dictionary=True)

    table_names = []
    try:
        table_names = [query.lower().split("from")[1].split()[0] for query in queries] 
    except Exception as e:
        print("Invalid query!", f"Error: {e}")

    data = {}
    for table_name, query in zip(table_names, queries):
        cursor.execute(query)
        data[table_name] = cursor.fetchall()
    
    cursor.close()
    connection.close()

    return data, table_names

In [3]:
SELLECT_DATA_FROM_ARTICLES = "SELECT * FROM thong_tin_quy_dinh"

data, table_names = load_data_from_database([SELLECT_DATA_FROM_ARTICLES])

for table_name in table_names:
    print(f" + Table: {table_name}, Number of rows: {len(data[table_name])}")
    for row in data[table_name][:5]:
        print(row)
    print('-'*50)

 + Table: thong_tin_quy_dinh, Number of rows: 40
{'id': 1, 'title': 'Phạm vi và đối tượng áp dụng', 'content': '1. Quy chế này quy định tổ chức và quản lý đào tạo trình độ đại học hệ chính quy theo hệ thống tín chỉ tại Trường Đại học Tôn Đức Thắng, bao gồm: Chương trình đào tạo, tổ chức đào tạo, đánh giá kết quả học tập, xét và công nhận tốt nghiệp và những quy định khác.\n2. Quy chế này áp dụng đối với sinh viên chương trình đào tạo tiêu chuẩn, chương trình đào tạo chất lượng cao, chương trình đào tạo giảng dạy bằng tiếng Anh hệ chính quy trình độ đại học tại Trường Đại học Tôn Đức Thắng (sau đây gọi tắt là Trường) theo hình thức tích lũy tín chỉ.'}
{'id': 2, 'title': 'Chương trình đào tạo', 'content': '1. Chương trình đào tạo (CTĐT) được xây dựng theo đơn vị tín chỉ, cấu trúc từ các môn học hoặc học phần (gọi chung là môn học), trong đó phải có đủ các môn học bắt buộc và đáp ứng chuẩn chương trình đào tạo theo quy định hiện hành của Bộ Giáo dục và Đào tạo (GD&ĐT). Trong trường hợp đà

### Chuyển dữ liệu từ database sang dataframe

In [4]:
import pandas as pd

df = pd.DataFrame(data['thong_tin_quy_dinh'])
# df.drop(columns=['id'], inplace=True)
df.head()

,id,title,content
0,1,Phạm vi và đối tượng áp dụng,1. Quy chế này quy định tổ chức và quản lý đào...
1,2,Chương trình đào tạo,1. Chương trình đào tạo (CTĐT) được xây dựng t...
2,3,Phương thức tổ chức đào tạo và hình thức đào tạo,1. Trường tổ chức đào tạo theo phương thức đào...
3,4,"Tín chỉ, Môn học trong Chương trình đào tạo",1. Tín chỉ được sử dụng để tính khối lượng học...
4,5,Thời gian đào tạo,1. Thời gian đào tạo theo kế hoạch học tập chu...


In [5]:
len(min(df['content'], key=len)), len(max(df['content'], key=len))

(218, 6920)

In [6]:
(min(df['content'], key=len)), (max(df['content'], key=len))

('Những sinh viên không đủ điều kiện cấp bằng tốt nghiệp được bảo lưu các môn học có kết quả từ 5,0 trở lên. Sinh viên không tốt nghiệp được cấp chứng nhận các môn học đã tích lũy trong chương trình đào tạo của Trường.',
 '1. "Không đủ điều kiện dự thi" hay “Điểm F” là hình thức xử lý các sinh viên vì một trong các lý do sau:\n\xa0 \xa0 a. Không thực hiện đầy đủ các phần bắt buộc của môn học và các yêu cầu theo quy định của môn học đặc thù (Cơ sở tin học, Tiếng Anh, Giáo dục thể chất, Giáo dục quốc phòng, Thực hành/Thí nghiệm, phần bài tập lớn, báo cáo, học phần nghề nghiệp...)\n\xa0 \xa0 b. Vi phạm nghiêm trọng kỷ luật học tập, nội quy học đường, gian dối trong học tập.\n\xa0 \xa0 c. Không đảm bảo tối thiểu 80% yêu cầu bắt buộc học tập trên lớp và ở nhà được quy định cho từng môn học\n\xa0 \xa0 d. Không hoàn thành nghĩa vụ học phí theo quy định. \nTrong buổi học đầu tiên của môn học, cán bộ giảng dạy công bố cho sinh viên quy định về các phần bắt buộc theo điểm (a) và (c) tại khoản 

## Chunking data 
Chia dữ liệu

In [7]:
! pip install langchain

In [8]:
from langchain.document_loaders.dataframe import DataFrameLoader

loader = DataFrameLoader(data_frame=df, page_content_column='content') # page_content_column is the column name that contains the text data

tdtu_data = loader.load()


tdtu_data[:1]

[Document(metadata={'id': 1, 'title': 'Phạm vi và đối tượng áp dụng'}, page_content='1. Quy chế này quy định tổ chức và quản lý đào tạo trình độ đại học hệ chính quy theo hệ thống tín chỉ tại Trường Đại học Tôn Đức Thắng, bao gồm: Chương trình đào tạo, tổ chức đào tạo, đánh giá kết quả học tập, xét và công nhận tốt nghiệp và những quy định khác.\n2. Quy chế này áp dụng đối với sinh viên chương trình đào tạo tiêu chuẩn, chương trình đào tạo chất lượng cao, chương trình đào tạo giảng dạy bằng tiếng Anh hệ chính quy trình độ đại học tại Trường Đại học Tôn Đức Thắng (sau đây gọi tắt là Trường) theo hình thức tích lũy tín chỉ.')]

In [9]:
# Chunking data (mỗi bài 1 chunk)
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    length_function=len
)

tdtu_doc = splitter.transform_documents(tdtu_data)
print(f"Number of chunks: {len(tdtu_doc)}")
tdtu_doc[:5]

Number of chunks: 181


[Document(metadata={'id': 1, 'title': 'Phạm vi và đối tượng áp dụng'}, page_content='1. Quy chế này quy định tổ chức và quản lý đào tạo trình độ đại học hệ chính quy theo hệ thống tín chỉ tại Trường Đại học Tôn Đức Thắng, bao gồm: Chương trình đào tạo, tổ chức đào tạo, đánh giá kết quả học tập, xét và công nhận tốt nghiệp và những quy định khác.'),
 Document(metadata={'id': 1, 'title': 'Phạm vi và đối tượng áp dụng'}, page_content='2. Quy chế này áp dụng đối với sinh viên chương trình đào tạo tiêu chuẩn, chương trình đào tạo chất lượng cao, chương trình đào tạo giảng dạy bằng tiếng Anh hệ chính quy trình độ đại học tại Trường Đại học Tôn Đức Thắng (sau đây gọi tắt là Trường) theo hình thức tích lũy tín chỉ.'),
 Document(metadata={'id': 2, 'title': 'Chương trình đào tạo'}, page_content='1. Chương trình đào tạo (CTĐT) được xây dựng theo đơn vị tín chỉ, cấu trúc từ các môn học hoặc học phần (gọi chung là môn học), trong đó phải có đủ các môn học bắt buộc và đáp ứng chuẩn chương trình đào 

## Embedding data
Using `halong_embedding` to embed data

In [10]:
! pip install -U faiss-cpu

In [11]:
# Load model
from langchain.embeddings import HuggingFaceEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores import FAISS

local_store = LocalFileStore('./cache')

emb_id = "hiieu/halong_embedding"
# Load model from huggingface
core_embeder = HuggingFaceEmbeddings(model_name=emb_id)
# Cache model to local storage
embeder = CacheBackedEmbeddings.from_bytes_store(core_embeder, local_store, namespace=emb_id)

# Create vector store
vector_store = FAISS.from_documents(tdtu_doc, embeder)

C:\Users\Phu Nguyen\AppData\Local\Temp\ipykernel_22520\2335888982.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  core_embeder = HuggingFaceEmbeddings(model_name=emb_id)
c:\Users\Phu Nguyen\AppData\Roaming\anaconda3\envs\cuda_conda_env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## Test with a sample query

In [12]:
query = "Cách đăng ký học lại" # "Cách tính điểm khi đăng ký học lại?"
query_vector = embeder.embed_query(query)
results = vector_store.similarity_search_by_vector(query_vector, k=3)

for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.metadata['id']}|{doc.metadata['title']}")
    print(f"Answer: {doc.page_content}")
    print('-'*50)

Result 1: 31|Xét và công nhận tốt nghiệp
Answer: - Đăng ký dự xét tốt nghiệp trên Cổng thông tin sinh viên.
      - Hoàn thành nghĩa vụ học phí và các nghĩa vụ khác với Trường.
--------------------------------------------------
Result 2: 11|Đăng ký Kế hoạch học tập
Answer: học cho sinh viên. Sinh viên phải thực hiện đăng ký môn học dựa trên KHHT đã đăng ký trước khi học kỳ mới bắt đầu (sinh viên mới trúng tuyển không phải đăng ký môn học cho học kỳ đầu tiên của khóa học).
--------------------------------------------------
Result 3: 15|Đăng ký học lại
Answer: 2. Đối với các môn học đã tích lũy, sinh viên có thể đăng ký học lại để cải thiện kết quả. Điểm cao nhất trong các lần học sẽ là kết quả chính thức của môn học đó. Điểm cải thiện không được tính trong quá trình xét học bổng khuyến khích học tập, xét khen thưởng học tập, xét khen thưởng cho sinh viên tốt nghiệp giỏi/xuất sắc theo Quy chế Công tác sinh viên của Trường.
--------------------------------------------------


### Save the vector store to database

In [13]:
len(pickle.dumps(vector_store.docstore))

107296

In [14]:
a = [1, 2, 3]
if a[3:6]:
    print("True")

In [21]:
import faiss
import pickle
from tqdm import tqdm

def split_bytes(data, chunk_size):
    return [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]

def save_model_to_database(vector_store: FAISS, chunk_size: int = 10000):
    connection = connect()
    cursor = connection.cursor()
    cursor.execute(f"DROP TABLE IF EXISTS faiss_index")
    # Create table
    cursor.execute(f"""CREATE TABLE IF NOT EXISTS faiss_index 
                   (id INT AUTO_INCREMENT PRIMARY KEY, 
                   vector BLOB, 
                   docstore BLOB,
                   metadata BLOB)""")
    connection.commit()
    try:
        get_elmt = lambda x, i: x[i] if i < len(x) else b''
        # Save model
        # get index and metadata from vector store (the core of FAISS)
        index = vector_store.index
        metadata = vector_store.index_to_docstore_id
        docstore = vector_store.docstore
        # serialize index and metadata and convert them to bytes
        index_bytes = faiss.serialize_index(index) # type: numpy.ndarray; length=556077
        index_bytes = index_bytes.tobytes() # type: bytes
        metadata_bytes = pickle.dumps(metadata) # type: bytes
        docstore_bytes = pickle.dumps(docstore) # type: bytes
        # Cuz the length of index_bytes is too long, we need to split it into smaller chunks
        index_bytes_chunks = split_bytes(index_bytes, chunk_size)
        mt_bytes_chunks = split_bytes(metadata_bytes, chunk_size)
        ds_bytes_chunks = split_bytes(docstore_bytes, chunk_size)
        # get the max length of chunks
        max_nchunks = max([len(index_bytes_chunks), len(mt_bytes_chunks), len(ds_bytes_chunks)])
        print(f"Number of rows: {max_nchunks}")
        for i in tqdm(range(max_nchunks)):
            index_chunk = get_elmt(index_bytes_chunks, i)
            meta_chunk = get_elmt(mt_bytes_chunks, i)
            ds_chunk = get_elmt(ds_bytes_chunks, i)
            
            # save to database
            cursor.execute("""INSERT INTO faiss_index (vector, docstore, metadata)
                            VALUES (%s, %s, %s)""", (index_chunk, ds_chunk, meta_chunk))

        connection.commit()
        print("Model saved successfully")
    except Exception as e:
        print(f"Error: {e}")
        
    cursor.close()
    connection.close()

save_model_to_database(vector_store)

Number of rows: 56


100%|██████████| 56/56 [00:02<00:00, 19.73it/s]

Model saved successfully


In [23]:
data_faiss, _ = load_data_from_database(["SELECT * FROM faiss_index"])
data_faiss['faiss_index']

[{'id': 1,
  'vector': b'IxF2\x00\x03\x00\x00\xb5\x00\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x00\x00\x00\x00\x00\x00\x10\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x1f\x02\x00\x00\x00\x00\x00e\xf0J\xbd$\t\xc8\xbc\xb0\xb9\x14=Z\x8dH=9\x8f\xa4<\x89\xe8C\xbd\xc4q\x92\xbc\x8a\xd0\x05<\\/><\x14\xfe\xe0<\xed\xe1w=\x83\x1a\t\xbdY"\x8c\xbc\xae`\xe8;\x15xk<\x10Y\xea;\xb3\xa0\x85\xbd$\x02)=\xe0\xc0\x88\xbd\x1a[\x82=uE\x15<\xf4\xe4M\xbd\xdb\xa6\xc1\xbc\x08\xa1\xb9=\xb2\xdd\\=3\x1e\xc3\xbc$\xe5\xc5;\x16M\x8a\xbc9\xd5\x03\xbc\xf3\x92)\xbdfP];\x84A\x12<\xbd\x80O\xbdU8\xff\xbc\x86\xe8\x88;\xed\x11\xc1\xbc\x1dT\x8d;P&\x15=zg\x90\xbc\x02[x=.g\xf4\xbcE\x1f\x19\xbd%w\xc6\xbcz\xc2\x81\xbd\x1b\x9dU\xbd\x8b\x9f#\xbd\x1dU\xda<\xaa\xe7J\xbd\x05\xdbH<H\xd3q\xbb\xb6\x9dX\xbc\xa4y\xcb=m2\x9c<\xe8\xc4\xbe<\xfd\x8e\xb6\xbd\xc1\xf0\xd1:\x14B\x98\xbc\xe7\xcb=\xbd\xd5\xdb==g,\x97=\ns\x99\xbd\xb1\x81\x8d=\xe41\x1f\xbd\xee>\x99\xbc\xba\ry\xbd\x99\xfe\x19\xbaI\x0b\xab<64V\xb94\xa6\xe1<pw!\xbdv\x17f\xbd\xdfpy\xbd\xe9\x

In [25]:
# Khôi phục model từ database
import re
def load_model_from_dict(d: list[dict], embeder) -> FAISS:
    index_chunks = [row['vector'] for row in d]
    index_bytes = b''.join(index_chunks)

    docstore_chunks = [row['docstore'] for row in d]
    docstore_bytes = b''.join(docstore_chunks)

    # deserialize index/docstore with index converted from bytes to numpy array
    index = faiss.deserialize_index(np.frombuffer(index_bytes, dtype=np.uint8)) # shape: (556077,)
    docstore = pickle.loads(np.frombuffer(docstore_bytes, dtype=np.uint8))
    metadata = pickle.loads(d[0]['metadata'])

    vector_store = FAISS(index=index, index_to_docstore_id=metadata, embedding_function=embeder, docstore=docstore)

    return vector_store

data_faiss, _ = load_data_from_database(["SELECT * FROM faiss_index"])
loaded_vector = load_model_from_dict(data_faiss['faiss_index'], embeder)

In [34]:
query = "Trường Đại học Tôn Đức Thắng thành lập năm nào?"
query_vector = embeder.embed_query(query)
results = loaded_vector.similarity_search_by_vector(query_vector, k=3)

for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.metadata['id']}|{doc.metadata['title']}")
    print(f"Answer: {doc.page_content}")
    print('-'*50)

Result 1: 38|Lịch sử hình thành
Answer: Tiền thân của Trường Đại học Tôn Đức Thắng (Ton Duc Thang University: TDTU) là Trường Đại học Công nghệ Dân lập Tôn Đức Thắng, thành lập theo Quyết định 787/TTg-QĐ ngày 24/9/1997 của Thủ tướng Chính phủ. Trường do Liên đoàn lao động Thành phố Hồ Chí Minh sáng lập và quản lý thông qua Hội đồng quản trị do Chủ tịch Liên đoàn lao động Thành phố đương nhiệm làm chủ tịch.
--------------------------------------------------
Result 2: 40|Định hướng phát triển
Answer: Trường Đại học Tôn Đức Thắng có trách nhiệm phát triển con người; phụng sự đất nước Việt Nam; giáo dục nguồn nhân lực chất lượng cao cho Thành phố Hồ Chí Minh và cả nước; trong đó, có sự chú trọng đào tạo đội ngũ công nhân - lao động; thực hiện nghiên cứu ứng dụng, nghiên cứu khoa học ngày càng hiệu quả để thúc đẩy đất nước phát triển trong dài hạn; cam kết cống hiến ngày càng nhiều và tốt hơn cho một Việt Nam phồn vinh, ổn định và bền vững; cũng như góp phần tạo dựng một thế giới văn minh v